# FusionNav V2: causal relative-motion pilot — PREPARED, NOT TRAINED

Запускать только после отдельного указания пользователя. `RUN_TRAINING=False` по умолчанию.
Подключите dataset **mrisdal/zurich-urban-micro-aerial-vehicle, version 2**: нужны только
RawAccel.csv, RawGyro.csv, OnboardGPS.csv и GroundTruthAGL.csv; хеши будут проверены.
Входы R7/checkpoints для нового обучения не используются. Никакие пакеты не устанавливаются.
Нужны NumPy, pandas, PyTorch и pyproj в окружении Kaggle. GPU необязателен.

Один seed, три маленькие модели: одинаковые GRU full/gps_only (нулевая IMU-ветвь),
LSTM full; 8 фиксированных epochs. Контроль + 9 заранее заданных validation outages.
Baselines: held-GNSS и причинный constant-velocity. Test не оценивается.
Primary target/metric — относительное движение камеры от последнего разрешённого fix,
а не абсолютная поправка GNSS→GT. Absolute raw error сохраняется отдельно.

Единые UTM32N оси и общий origin; gyro только из прошлого; dt из timestamps.
Позиция не подгоняется по GT. Точный высотный datum GT и camera/antenna extrinsic остаются
ограничениями: motion-labels устраняют постоянное начальное смещение только в метрике/loss,
не передавая GT модели. Это исследовательский pilot, не физически сертифицированный INS.


In [ ]:
from pathlib import Path
import sys
RUN_TRAINING = False  # Change only after a separate explicit user command to run the pilot.
CODE_ROOT = Path('/kaggle/working/v2_motion_pilot_code')
OUT_DIR = Path('/kaggle/working/outputs/v2_motion_pilot')
RAW_DIR = None


In [ ]:
import hashlib
SOURCES = {'src/diagnostics/__init__.py': '"""Isolated, inference-only diagnostics; the R7 pipeline is unchanged."""\n', 'src/diagnostics/clean_outages.py': '"""Clean outages on the existing R7 coordinate/time representation.\n\nNo training, GT-dependent input construction, alignment, or filesystem writes.\nAvailability is a link-level flag, NOT a claim that a new fix arrived. The\nseparate update mask and source indices make that distinction auditable.\n"""\nimport hashlib\n\nimport numpy as np\n\n\ndef _timeline(t):\n    t = np.asarray(t, dtype=np.float64)\n    if t.ndim != 1 or not len(t) or not np.isfinite(t).all() or (np.diff(t) <= 0).any():\n        raise ValueError("timestamps must be finite, nonempty and strictly increasing")\n    return t\n\n\ndef false_runs(available):\n    """Half-open index intervals in the FINAL mask, independent of block order."""\n    missing = ~np.asarray(available, dtype=bool)\n    edges = np.diff(np.r_[False, missing, False].astype(np.int8))\n    return list(zip(np.flatnonzero(edges == 1).tolist(),\n                    np.flatnonzero(edges == -1).tolist()))\n\n\ndef availability_from_blocks(n, blocks):\n    available = np.ones(n, dtype=bool)\n    for start, stop in blocks:\n        if not (0 <= start < stop <= n):\n            raise ValueError("blocks must be nonempty half-open intervals within the record")\n        available[start:stop] = False\n    return available\n\n\ndef mask_sha256(available):\n    """Hash uint8 availability bytes, one byte per common-grid sample."""\n    return hashlib.sha256(np.asarray(available, dtype=np.uint8).tobytes()).hexdigest()\n\n\ndef common_validation_indices(t_imu, t_gt, train_fraction, val_fraction, window):\n    """Split first, restrict to original GT support, then trim incomplete windows.\n\n    All methods use these SAME indices. No GT extrapolation, no concatenation\n    across holes, no carry-over from train/test. Windows start at common[0].\n    """\n    t = _timeline(t_imu)\n    g = _timeline(t_gt)\n    if window < 1 or not (0 < train_fraction < 1 and 0 < val_fraction < 1 - train_fraction):\n        raise ValueError("invalid split/window")\n    start = int(len(t) * train_fraction)\n    stop = start + int(len(t) * val_fraction)\n    raw = np.arange(start, stop, dtype=np.int64)\n    supported = raw[(t[raw] >= g[0]) & (t[raw] <= g[-1])]\n    if len(supported) and (np.diff(supported) != 1).any():\n        raise ValueError("noncontiguous supported validation region")\n    n = len(supported) // window * window\n    if not n:\n        raise ValueError("validation has no complete supported window")\n    common = supported[:n]\n    return common, {\n        "split": "validation", "original_split_indices_half_open": [start, stop],\n        "original_split_n": len(raw), "gt_support_s": [float(g[0]), float(g[-1])],\n        "excluded_outside_gt_support_n": len(raw) - len(supported),\n        "discarded_incomplete_window_n": len(supported) - n,\n        "common_indices_half_open": [int(common[0]), int(common[-1]) + 1],\n        "common_n": n, "window": window, "stride": window, "window_count": n // window,\n        "common_first_timestamp_s": float(t[common[0]]),\n        "common_last_timestamp_s": float(t[common[-1]]),\n    }\n\n\ndef make_scenarios(t, protocol):\n    """Choose starts from timestamps only; never inspect positions/errors.\n\n    Three fractions of the admissible onset interval are snapped forward to a\n    grid point. Every duration uses the same starts, in separate full-trace\n    scenarios. [onset, cutoff) is unavailable. RNG is explicitly unused.\n    """\n    t = _timeline(t)\n    durations = protocol["durations_s"]\n    fractions = protocol["start_fractions"]\n    warmup, recovery = protocol["warmup_s"], protocol["recovery_s"]\n    if min(durations) <= 0 or warmup <= 0 or recovery <= 0:\n        raise ValueError("positive durations, warmup and recovery required")\n    if any(not 0 <= f <= 1 for f in fractions):\n        raise ValueError("start fractions must be in [0, 1]")\n    lo, hi = t[0] + warmup, t[-1] - recovery - max(durations)\n    if hi <= lo:\n        raise ValueError("validation too short for the fixed protocol")\n    starts = [int(np.searchsorted(t, lo + f * (hi - lo), side="left")) for f in fractions]\n    if len(set(starts)) != len(starts):\n        raise ValueError("not enough distinct starts; do not silently change protocol")\n    scenarios = [{"scenario_id": "val_control", "split": "validation",\n                  "episode_id": None, "requested_duration_s": 0,\n                  "source_blocked_intervals_s": [], "unavailable_index_runs": [],\n                  "n_unavailable": 0, "metric_scope": "whole_common_trace"}]\n    for episode, start in enumerate(starts, 1):\n        for duration in durations:\n            onset = float(t[start])\n            cutoff = onset + duration\n            stop = int(np.searchsorted(t, cutoff, side="left"))\n            if start == 0 or stop >= len(t) or t[-1] - t[stop] < recovery:\n                raise ValueError("episode lacks warmup/recovery; protocol cannot be used")\n            scenarios.append({\n                "scenario_id": f"val_e{episode}_d{duration:g}s", "split": "validation",\n                "episode_id": episode, "requested_duration_s": duration,\n                "target_onset_s": float(lo + fractions[episode - 1] * (hi - lo)),\n                "onset_s": onset, "requested_cutoff_s": cutoff,\n                "last_unavailable_s": float(t[stop - 1]),\n                "first_link_available_after_s": float(t[stop]),\n                "observed_mask_interval_s": float(t[stop] - t[start]),\n                "unavailable_sample_span_s": float(t[stop - 1] - t[start]),\n                "available_history_span_s": float(t[start] - t[0]),\n                "recovery_span_s": float(t[-1] - t[stop]),\n                "source_blocked_intervals_s": [[onset, cutoff]],\n                "unavailable_index_runs": [[start, stop]], "n_unavailable": stop - start,\n                "metric_scope": "outage_only",\n            })\n    for scenario in scenarios:\n        available = availability_from_blocks(len(t), scenario["unavailable_index_runs"])\n        scenario.update(n_common=len(t), mask_uint8_sha256=mask_sha256(available),\n                        generator_seed=protocol["generator_seed"], rng_used=False)\n    return scenarios\n\n\ndef clean_gnss(t, candidates, source_indices, source_timestamps, available,\n               blocked_intervals=None):\n    """Sequentially accept only legal candidate fixes after forming final mask.\n\n    Candidates are the unchanged R7 GPS resampling. Source timestamps/indices\n    refer to exported processed GPS rows, NOT verified physical receiver fixes.\n    During a false run, keep the last accepted value from BEFORE its onset.\n    Also reject a buffered candidate stamped inside a denied time interval on\n    recovery. No candidate at the first masked sample is accepted. The first\n    grid sample must have a legal fix; this protocol provides explicit warmup.\n    """\n    t = _timeline(t)\n    candidates = np.asarray(candidates)\n    ids = np.asarray(source_indices, dtype=np.int64)\n    st = np.asarray(source_timestamps, dtype=np.float64)\n    available = np.asarray(available, dtype=bool)\n    if (candidates.shape != (len(t), 3) or ids.shape != t.shape\n            or st.shape != t.shape or available.shape != t.shape):\n        raise ValueError("GNSS inputs have incompatible shapes")\n    if not np.isfinite(candidates).all() or not np.isfinite(st).all() or (ids < 0).any():\n        raise ValueError("invalid candidate fixes")\n    if (st > t).any():\n        raise ValueError("R7 resampling selected a future GPS source; refusing silent repair")\n    if blocked_intervals is None:\n        blocked_intervals = [(t[a], t[b] if b < len(t) else np.inf)\n                             for a, b in false_runs(available)]\n    forbidden_source = np.zeros(len(t), dtype=bool)\n    denied_grid = np.zeros(len(t), dtype=bool)\n    for a, b in blocked_intervals:\n        if not a < b:\n            raise ValueError("invalid denied time interval")\n        forbidden_source |= (st >= a) & (st < b)\n        denied_grid |= (t >= a) & (t < b)\n    if not np.array_equal(denied_grid, ~available):\n        raise ValueError("source denial intervals must match the final grid mask")\n    held = np.empty_like(candidates)\n    held_ids = np.empty(len(t), dtype=np.int64)\n    held_st = np.empty(len(t), dtype=np.float64)\n    accepted_at = np.empty(len(t), dtype=np.int64)\n    updates = np.zeros(len(t), dtype=bool)\n    last = None\n    for i in range(len(t)):\n        if available[i] and not forbidden_source[i]:\n            if last is not None and (ids[i] < ids[last] or st[i] < st[last]):\n                raise ValueError("selected source order reversed; refusing silent resampling change")\n            if last is None or ids[i] != ids[last]:\n                last = i\n                updates[i] = True\n            elif not np.array_equal(candidates[i], candidates[last]) or st[i] != st[last]:\n                raise ValueError("same source index has inconsistent timestamp/position")\n        if last is None:\n            raise ValueError("no legal pre-outage fix; provide available GNSS history, never GT")\n        held[i], held_ids[i], held_st[i], accepted_at[i] = candidates[last], ids[last], st[last], last\n    return {"held_gnss": held, "availability_mask": available.copy(),\n            "gnss_update_mask": updates, "held_source_index": held_ids,\n            "held_source_timestamp_s": held_st, "accepted_at_grid_index": accepted_at,\n            "fix_age_s": t - held_st,\n            "rejected_buffered_source_mask": available & forbidden_source}\n\n\ndef constant_velocity_gnss(t, clean, history_s=5.0, min_span_s=1.0):\n    """Frozen causal least-squares XYZ velocity from accepted pre-onset fixes.\n\n    Unique source rows within the trailing history_s (source time), at least\n    two distinct timestamps spanning min_span_s. Otherwise v=0. Anchor at the\n    last legal position AND its source time, extrapolate only in mask=False.\n    Link recovery returns to held-GNSS; no fit to GT or later samples.\n    """\n    t = _timeline(t)\n    if history_s <= 0 or min_span_s <= 0:\n        raise ValueError("positive velocity history/span required")\n    held = clean["held_gnss"]\n    st = clean["held_source_timestamp_s"]\n    update = clean["gnss_update_mask"]\n    pred = held.astype(np.float64).copy()\n    records = []\n    for a, b in false_runs(clean["availability_mask"]):\n        if a == 0:\n            raise ValueError("CV requires a legal pre-onset anchor")\n        hist = np.flatnonzero(update & (np.arange(len(t)) < a)\n                              & (st >= t[a] - history_s) & (st < t[a]))\n        # An exported stream can contain multiple rows with the same timestamp.\n        # Keep the last accepted row per timestamp; do not overweight duplicates.\n        if len(hist):\n            hist = hist[np.r_[np.diff(st[hist]) != 0, True]]\n        velocity = np.zeros(3, dtype=np.float64)\n        span = float(st[hist[-1]] - st[hist[0]]) if len(hist) else 0.0\n        fallback = len(hist) < 2 or span < min_span_s\n        if not fallback:\n            x = st[hist] - st[hist].mean()\n            y = held[hist].astype(np.float64)\n            velocity = (x[:, None] * (y - y.mean(axis=0))).sum(axis=0) / (x @ x)\n        pred[a:b] = held[a - 1] + (t[a:b] - st[a - 1])[:, None] * velocity\n        records.append({"indices_half_open": [a, b], "velocity_m_s": velocity.tolist(),\n                        "history_source_indices": clean["held_source_index"][hist].tolist(),\n                        "history_grid_indices": hist.tolist(), "history_span_s": span,\n                        "history_source_timestamps_s": st[hist].tolist(),\n                        "anchor_source_index": int(clean["held_source_index"][a - 1]),\n                        "anchor_source_timestamp_s": float(st[a - 1]),\n                        "anchor_position_m": held[a - 1].tolist(),\n                        "zero_velocity_fallback": bool(fallback)})\n    return pred, records\n\n\ndef raw_errors(pred, gt):\n    pred, gt = np.asarray(pred, dtype=np.float64), np.asarray(gt, dtype=np.float64)\n    if pred.shape != gt.shape or pred.ndim != 2 or pred.shape[1] != 3:\n        raise ValueError("pred and GT must be matching N x 3 arrays")\n    if not np.isfinite(pred).all() or not np.isfinite(gt).all():\n        raise ValueError("nonfinite prediction/GT")\n    delta = pred - gt\n    return {"error_xyz_m": delta, "error_3d_m": np.linalg.norm(delta, axis=1),\n            "error_horizontal_m": np.linalg.norm(delta[:, :2], axis=1),\n            "error_vertical_signed_m": delta[:, 2]}\n\n\ndef raw_metrics(pred, gt, available):\n    """Sample-weighted RAW errors; no rigid alignment, fit, SD, or CI.\n\n    All unavailable points for an outage scenario (one episode here); the whole\n    common trace for control. Final errors always mean last unavailable sample,\n    and are None for control. Horizontal=existing first two project axes.\n    """\n    errors = raw_errors(pred, gt)\n    available = np.asarray(available, dtype=bool)\n    if available.shape != (len(pred),):\n        raise ValueError("metric mask shape mismatch")\n    missing = np.flatnonzero(~available)\n    chosen = missing if len(missing) else np.arange(len(pred))\n    if not len(chosen):\n        raise ValueError("empty metric support")\n    d = errors["error_xyz_m"][chosen]\n    row = {"metric_scope": "outage_only" if len(missing) else "whole_common_trace",\n           "n_evaluated": len(chosen), "rmse_3d_m": float(np.sqrt(np.mean(np.sum(d * d, axis=1)))),\n           "rmse_horizontal_m": float(np.sqrt(np.mean(np.sum(d[:, :2] ** 2, axis=1)))),\n           "rmse_vertical_m": float(np.sqrt(np.mean(d[:, 2] ** 2))),\n           "final_unavailable_error_3d_m": None, "final_unavailable_error_horizontal_m": None,\n           "final_unavailable_abs_vertical_m": None}\n    if len(missing):\n        last = missing[-1]\n        row.update(final_unavailable_error_3d_m=float(errors["error_3d_m"][last]),\n                   final_unavailable_error_horizontal_m=float(errors["error_horizontal_m"][last]),\n                   final_unavailable_abs_vertical_m=float(abs(errors["error_vertical_signed_m"][last])))\n    return row\n', 'src/diagnostics/next_stage.py': '"""Numerical diagnostics only; no training and no changes to the R7 pipeline."""\nimport numpy as np\n\n\ndef rmse_xyz(pred, target):\n    e = np.asarray(pred, float) - np.asarray(target, float)\n    if not len(e):\n        return {k: None for k in (\'rmse_3d_m\', \'rmse_horizontal_m\', \'rmse_vertical_m\')}\n    return {\'rmse_3d_m\': float(np.sqrt(np.mean(np.sum(e*e, axis=1)))),\n            \'rmse_horizontal_m\': float(np.sqrt(np.mean(np.sum(e[:, :2]**2, axis=1)))),\n            \'rmse_vertical_m\': float(np.sqrt(np.mean(e[:, 2]**2)))}\n\n\ndef fit_frame(x, y, kind):\n    """Return column-vector convention y = scale * R @ x + translation.\n\n    caller must explicitly identify fitting/evaluation split. A validation fit\n    is an in-sample oracle diagnostic, never an independent evaluation.\n    """\n    x, y = np.asarray(x, float), np.asarray(y, float)\n    mx, my = x.mean(0), y.mean(0)\n    rotation, scale = np.eye(3), 1.\n    if kind == \'raw\':\n        translation = np.zeros(3)\n    else:\n        if kind != \'translation\':\n            dimensions = 2 if kind == \'se2_vertical\' else 3\n            a, b = (x-mx)[:, :dimensions], (y-my)[:, :dimensions]\n            u, singular, vt = np.linalg.svd(np.einsum(\'ni,nj->ij\',a,b))\n            sign = np.ones(dimensions)\n            sign[-1] = np.linalg.det(vt.T @ u.T)\n            rotation[:dimensions, :dimensions] = vt.T @ np.diag(sign) @ u.T\n            if kind == \'similarity_diagnostic\':\n                scale = float((singular*sign).sum() / np.sum(a*a))\n            elif kind not in (\'se2_vertical\', \'se3\'):\n                raise ValueError(kind)\n        translation = my-scale*(rotation @ mx)\n    return {\'kind\': kind, \'rotation\': rotation.tolist(), \'translation_m\': translation.tolist(),\n            \'scale\': scale, \'yaw_deg\': float(np.degrees(np.arctan2(rotation[1,0],rotation[0,0]))),\n            \'rotation_angle_deg\': float(np.degrees(np.arccos(np.clip((np.trace(rotation)-1)/2,-1,1))))}\n\n\ndef apply_frame(x, fitted):\n    return fitted[\'scale\'] * np.einsum(\'ni,ji->nj\',np.asarray(x),np.array(fitted[\'rotation\'])) + fitted[\'translation_m\']\n\n\ndef utm32_wgs84(lat_deg, lon_deg):\n    """WGS84 transverse Mercator series, zone 32N, restricted to Zurich.\n\n    This analytic diagnostic is cross-checked against the dataset\'s UTM GPS\n    columns. V2 notebook uses pyproj for production projection instead.\n    """\n    lat = np.deg2rad(lat_deg)\n    lon = np.deg2rad(lon_deg)\n    a, f, k = 6378137., 1/298.257223563, .9996\n    e2 = f*(2-f)\n    ep2 = e2/(1-e2)\n    n = a/np.sqrt(1-e2*np.sin(lat)**2)\n    t = np.tan(lat)**2\n    c = ep2*np.cos(lat)**2\n    aa = np.cos(lat)*(lon-np.deg2rad(9.))\n    m = a*((1-e2/4-3*e2**2/64-5*e2**3/256)*lat\n           -(3*e2/8+3*e2**2/32+45*e2**3/1024)*np.sin(2*lat)\n           +(15*e2**2/256+45*e2**3/1024)*np.sin(4*lat)\n           -35*e2**3/3072*np.sin(6*lat))\n    east = 500000+k*n*(aa+(1-t+c)*aa**3/6+(5-18*t+t*t+72*c-58*ep2)*aa**5/120)\n    north = k*(m+n*np.tan(lat)*(aa*aa/2+(5-t+9*c+4*c*c)*aa**4/24\n                            +(61-58*t+t*t+600*c-330*ep2)*aa**6/720))\n    return np.column_stack((east, north))\n\n\ndef correction_stats(correction, t, chosen, window=192):\n    c = np.asarray(correction, float)\n    z = c[chosen]\n    if not len(z):\n        return {\'n\': 0}\n    consecutive = chosen[1:] & chosen[:-1]\n    d = np.diff(c, axis=0)[consecutive]\n    dt = np.diff(t)[consecutive]\n    lag = []\n    for axis in range(3):\n        x, y = c[:-1, axis][consecutive], c[1:, axis][consecutive]\n        lag.append(float(np.corrcoef(x, y)[0,1]) if len(x)>1 and x.std()>1e-12 and y.std()>1e-12 else None)\n    boundary = (np.arange(1,len(t)) % window == 0) & consecutive\n    within = (np.arange(1,len(t)) % window != 0) & consecutive\n    norm = np.linalg.norm(z,axis=1)\n    delta_norm = np.linalg.norm(np.diff(c,axis=0),axis=1)\n    return {\'n\': len(z), \'mean_xyz_m\':z.mean(0).tolist(), \'std_xyz_m_ddof0\':z.std(0).tolist(),\n            \'median_xyz_m\':np.median(z,axis=0).tolist(), \'min_xyz_m\':z.min(0).tolist(),\n            \'max_xyz_m\':z.max(0).tolist(), \'rms_norm_m\':float(np.sqrt(np.mean(norm**2))),\n            \'centered_rms_m\':float(np.sqrt(np.mean(np.sum((z-z.mean(0))**2,axis=1)))),\n            \'lag1_pearson_xyz\':lag,\n            \'adjacent_increment_rms_m\':float(np.sqrt(np.mean(np.sum(d*d,axis=1)))) if len(d) else None,\n            \'adjacent_derivative_rms_m_s\':float(np.sqrt(np.mean(np.sum((d/dt[:,None])**2,axis=1)))) if len(d) else None,\n            \'mean_window_boundary_jump_m\':float(delta_norm[boundary].mean()) if boundary.any() else None,\n            \'mean_within_window_jump_m\':float(delta_norm[within].mean()) if within.any() else None}\n\n\ndef pearson_features(correction, features, chosen):\n    result={}\n    for name, feature in features.items():\n        a=np.asarray(feature)[chosen]\n        result[name]=[]\n        for axis in range(3):\n            b=correction[chosen,axis]\n            r=float(np.corrcoef(a,b)[0,1]) if len(a)>2 and np.std(a)>1e-12 and np.std(b)>1e-12 else None\n            result[name].append(r)\n    return result\n\n\ndef future_gnss_exposure(t, available, source_t, cutoff, window=192):\n    """Potential post-recovery attention, not a causal effect measurement."""\n    available=np.asarray(available,bool)\n    exposed=np.zeros(len(t),bool)\n    fresh=np.zeros(len(t),bool)\n    for a in range(0,len(t),window):\n        b=min(a+window,len(t))\n        for i in range(a,b):\n            if not available[i]:\n                candidates=(np.arange(a,b)>i)&available[a:b]&(t[a:b]>=cutoff)\n                exposed[i]=candidates.any()\n                fresh[i]=(candidates&(source_t[a:b]>=cutoff)).any()\n    return exposed,fresh\n\n\ndef permute_imu_windows(imu, window, mode, seed):\n    """Paired accel+gyro row perturbation; no GNSS/GT input to this function."""\n    x=np.asarray(imu).reshape(-1,window,6).copy()\n    rng=np.random.default_rng(seed)\n    indices=[]\n    for w in range(len(x)):\n        p=rng.permutation(window) if mode==\'shuffle\' else np.arange(window)[::-1]\n        indices.append(p)\n        x[w]=x[w,p]\n    return x.reshape(-1,6), np.array(indices)\n', 'src/pilot_v2/__init__.py': '"""Prepared, untrained causal motion pilot; separate from R7."""\n', 'src/pilot_v2/data.py': '"""V2 data contract. No fitting GNSS->GT, pose inputs or GT-derived features."""\nimport hashlib\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom diagnostics.clean_outages import clean_gnss, false_runs\n\n\ndef backward_indices(source_t, query_t):\n    if np.any(np.diff(source_t)<0):\n        raise ValueError(\'sort source timestamps first\')\n    ids=np.searchsorted(source_t,query_t,side=\'right\')-1\n    if np.any(ids<0):\n        raise ValueError(\'no past measurement; do not clip to a future sample\')\n    return ids\n\n\ndef prepare(raw_dir,config):\n    """Read four small logs only, no images; project both streams into UTM32N.\n\n    pyproj is required, never installed automatically. Same GNSS-derived origin\n    subtracted from GPS and GT. Vertical values retained in documented metres;\n    no geoid conversion invented. Motion labels cancel constant height offsets.\n    """\n    raw_dir=Path(raw_dir)\n    for name,expected in config[\'raw_sha256\'].items():\n        if hashlib.sha256((raw_dir/name).read_bytes()).hexdigest()!=expected:\n            raise ValueError(\'unexpected raw data: \'+name)\n    from pyproj import Transformer\n    projection=Transformer.from_crs(\'EPSG:4326\',\'EPSG:32632\',always_xy=True)\n    frames={name:pd.read_csv(raw_dir/(name+\'.csv\')) for name in [\'RawAccel\',\'RawGyro\',\'OnboardGPS\',\'GroundTruthAGL\']}\n    for frame in frames.values():frame.columns=frame.columns.str.strip()\n    acc,gyro,gps,gt=[frames[name] for name in frames]\n    t=acc.Timpstemp.values.astype(float)*1e-6\n    if np.any(np.diff(t)<=0):raise ValueError(\'unexpected accel timeline\')\n    # Preserve original accel split indices. GPS joins GT by imgid BEFORE sorting.\n    gt[\'t\']=gt.imgid.map(gps.set_index(\'imgid\').Timpstemp)*1e-6\n    gt=gt.dropna(subset=[\'t\']).sort_values(\'t\',kind=\'stable\')\n    gyro=gyro.sort_values(\'Timpstemp\',kind=\'stable\').drop_duplicates(\'Timpstemp\',keep=\'last\')\n    tg=gyro.Timpstemp.values.astype(float)*1e-6\n    j=backward_indices(tg,t)\n    imu=np.column_stack([acc[[\'x\',\'y\',\'z\']].values,gyro[[\'x\',\'y\',\'z\']].values[j]]).astype(float)\n    gyro_age=t-tg[j]\n    gps=gps.sort_values(\'Timpstemp\',kind=\'stable\').drop_duplicates(\'Timpstemp\',keep=\'last\')\n    # Coordinate-change events are a documented proxy, not verified receiver epochs.\n    xyz=gps[[\'lat\',\'lon\',\'alt\']].values\n    changed=np.r_[True,np.any(np.diff(xyz,axis=0)!=0,axis=1)]\n    valid=(gps.fix_type.values>=3)&np.isfinite(xyz).all(axis=1)\n    event=gps.loc[changed&valid].copy()\n    event_t=event.Timpstemp.values.astype(float)*1e-6\n    east,north=projection.transform(event.lon.values,event.lat.values)\n    absolute=np.column_stack([east,north,event.alt.values])\n    origin=absolute[0].copy()\n    position=absolute-origin\n    gt_t=gt.t.values.astype(float)\n    gt_position=gt[[\'x_gt\',\'y_gt\',\'z_gt\']].values.astype(float)-origin\n    # Only train/validation region is returned. No test labels or inputs evaluated.\n    stop=config[\'validation_original_indices\'][1]\n    data={\'t\':t[:stop], \'imu\':imu[:stop], \'gyro_age\':gyro_age[:stop],\n          \'event_t\':event_t[event_t<=t[stop-1]],\'event_position\':position[event_t<=t[stop-1]],\n          \'gt_t\':gt_t[gt_t<=t[stop-1]],\'gt_position\':gt_position[gt_t<=t[stop-1]],\n          \'origin_utm_e_n_msl_m\':origin.tolist(),\n          \'frame_note\':\'same UTM grid and common origin; camera/antenna lever arm and exact GT height datum unresolved\'}\n    return data\n\n\ndef interpolate_gt(data,t):\n    if np.any(t<data[\'gt_t\'][0]) or np.any(t>data[\'gt_t\'][-1]):\n        raise ValueError(\'no GT extrapolation\')\n    return np.column_stack([np.interp(t,data[\'gt_t\'],data[\'gt_position\'][:,j]) for j in range(3)])\n\n\ndef fit_imu_scaler(imu,train_stop):\n    x=np.asarray(imu[:train_stop],float)\n    return {\'mean\':x.mean(0).tolist(),\'std\':np.maximum(x.std(0),1e-6).tolist()}\n\n\ndef build_inputs(data,indices,available,scaler,blocked_intervals):\n    """No GT read here. Actual dt; backwards gyro; no absolute position feature."""\n    t=data[\'t\'][indices]\n    event=backward_indices(data[\'event_t\'],t)\n    clean=clean_gnss(t,data[\'event_position\'][event],event,data[\'event_t\'][event],\n                     available,blocked_intervals)\n    held=clean[\'held_gnss\'];update=clean[\'gnss_update_mask\'];st=clean[\'held_source_timestamp_s\']\n    dt=np.r_[0.,np.diff(t)]\n    velocity=np.zeros((len(t),3));delta=np.zeros_like(velocity)\n    history=[];v=np.zeros(3)\n    for i in range(len(t)):\n        if update[i]:\n            if history:delta[i]=held[i]-held[history[-1]]\n            history.append(i)\n            history=[k for k in history if st[k]>=t[i]-5.]\n            if len(history)>=2 and st[history[-1]]-st[history[0]]>=1.:\n                x=st[history]-st[history].mean();y=held[history]\n                v=(x[:,None]*(y-y.mean(0))).sum(0)/(x@x)\n            else:v=np.zeros(3)\n        velocity[i]=v\n    # Freeze CV exactly at pre-onset value; history does not update in outage.\n    effective_dt=dt.copy()\n    for a,b in false_runs(available):\n        effective_dt[a]=t[a]-st[a-1]\n    imu=(data[\'imu\'][indices]-scaler[\'mean\'])/scaler[\'std\']\n    imu_features=np.column_stack([imu,data[\'gyro_age\'][indices],(dt>.15).astype(float)])\n    gnss_features=np.column_stack([delta,velocity,clean[\'fix_age_s\']/60.,\n                                   available.astype(float),update.astype(float),dt])\n    assert imu_features.shape[1]==8 and gnss_features.shape[1]==10\n    return {\'imu_features\':imu_features.astype(np.float32),\'gnss_features\':gnss_features.astype(np.float32),\n            \'velocity_prior\':velocity.astype(np.float32),\'integration_dt\':effective_dt.astype(np.float32),\n            \'t\':t,\'held\':held,\'availability\':np.asarray(available,bool),\'clean\':clean}\n\n\ndef motion_labels(data,inputs,label_period_s=1.):\n    """GT used ONLY as target/reference, never in build_inputs or rollout.\n\n    Native GT >=~1 Hz; sample relative-displacement loss about once per second,\n    not as independent 10 Hz ground-truth velocity observations.\n    """\n    t=inputs[\'t\'];target=np.zeros((len(t),3));chosen=np.zeros(len(t),bool)\n    gt=interpolate_gt(data,t)\n    for a,b in false_runs(inputs[\'availability\']):\n        anchor_t=inputs[\'clean\'][\'held_source_timestamp_s\'][a-1]\n        anchor_gt=interpolate_gt(data,np.array([anchor_t]))[0]\n        target[a:b]=gt[a:b]-anchor_gt\n        times=np.arange(t[a],t[b-1]+1e-9,label_period_s)\n        ids=np.unique(np.r_[np.searchsorted(t,times),b-1])\n        chosen[ids[(ids>=a)&(ids<b)]]=True\n    return target.astype(np.float32),chosen,gt\n\n\ndef baseline_predictions(inputs):\n    held=inputs[\'held\'].copy();cv=held.copy()\n    for a,b in false_runs(inputs[\'availability\']):\n        anchor=held[a-1]\n        st=inputs[\'clean\'][\'held_source_timestamp_s\'][a-1]\n        cv[a:b]=anchor+(inputs[\'t\'][a:b]-st)[:,None]*inputs[\'velocity_prior\'][a-1]\n    return held,cv\n', 'src/pilot_v2/model.py': '"""Compact causal recurrent motion model. Same parameter count for full/gps_only."""\nimport torch\nfrom torch import nn\n\n\nclass CausalMotion(nn.Module):\n    def __init__(self,hidden=32,ablation=\'full\',backbone=\'gru\'):\n        super().__init__()\n        self.ablation=ablation\n        self.imu_embed=nn.Sequential(nn.Linear(8,16),nn.Tanh())\n        self.gps_embed=nn.Sequential(nn.Linear(10,16),nn.Tanh())\n        layer=nn.GRU if backbone==\'gru\' else nn.LSTM\n        self.recurrent=layer(32,hidden,batch_first=True)\n        self.head=nn.Linear(hidden,3)\n        nn.init.zeros_(self.head.weight);nn.init.zeros_(self.head.bias)\n\n    def forward(self,imu,gnss,state=None):\n        if self.ablation==\'gps_only\':imu=torch.zeros_like(imu)\n        features=torch.cat([self.imu_embed(imu),self.gps_embed(gnss)],dim=-1)\n        h,state=self.recurrent(features,state)\n        return self.head(h),state\n\n\ndef rollout_motion(delta_velocity,velocity_prior,integration_dt,availability):\n    """Relative displacement since last fix. No GT or teacher forcing.\n\n    Independent outage segments; no artificial reset within an outage. At\n    availability return the position estimate is set to legal held-GNSS.\n    """\n    velocity=velocity_prior+delta_velocity\n    increments=velocity*integration_dt[...,None]\n    # Cumulative motion resets on available tokens; differentiable with torch.\n    total=torch.cumsum(increments,dim=1)\n    index=torch.arange(total.shape[1],device=total.device)[None].expand(total.shape[0],-1)\n    last=torch.cummax(torch.where(availability,index,torch.zeros_like(index)),dim=1).values\n    anchor=torch.gather(total,1,last[...,None].expand(-1,-1,3))\n    return torch.where(availability[...,None],torch.zeros_like(total),total-anchor)\n', 'src/pilot_v2/run.py': '"""Prepared pilot. Training requires explicit --run-training authorization.\n\nNo execution on import. Final-epoch checkpoints, one seed, no test split.\n"""\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport platform\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nfrom diagnostics.clean_outages import availability_from_blocks, false_runs, raw_metrics\nfrom pilot_v2.data import prepare, build_inputs, motion_labels, fit_imu_scaler, baseline_predictions\nfrom pilot_v2.model import CausalMotion, rollout_motion\n\n\ndef json_write(path,obj):\n    Path(path).write_text(json.dumps(obj,indent=2,allow_nan=False)+\'\\n\')\n\n\ndef csv_write(path,rows):\n    with Path(path).open(\'x\',newline=\'\') as stream:\n        writer=csv.DictWriter(stream,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)\n\n\ndef tensor(array,device,dtype=torch.float32):\n    return torch.as_tensor(array,dtype=dtype,device=device)\n\n\ndef training_specs(data,cfg,epoch):\n    w=cfg[\'train_window_samples\'];end=cfg[\'train_stop_index\']\n    rng=np.random.default_rng(cfg[\'seed\']+10000+epoch)\n    specs=[]\n    for start in range(0,end-w+1,cfg[\'train_stride_samples\']):\n        ids=np.arange(start,start+w);t=data[\'t\'][ids]\n        duration=float(rng.choice(cfg[\'outage_durations_s\']))\n        # Reserve two maximal sample gaps for snapping onset and recovery.\n        low=t[0]+cfg[\'train_warmup_s\'];high=t[-1]-duration-cfg[\'train_recovery_s\']-2*np.max(np.diff(t))\n        if high<=low:raise ValueError(\'training window too short for fixed outage curriculum\')\n        a=int(np.searchsorted(t,rng.uniform(low,high)));onset=float(t[a]);cutoff=onset+duration\n        b=int(np.searchsorted(t,cutoff))\n        if b>=len(t) or t[-1]-t[b]<cfg[\'train_recovery_s\']:\n            raise ValueError(\'outage has insufficient recovery\')\n        specs.append({\'indices\':ids,\'available\':availability_from_blocks(w,[(a,b)]),\n                      \'blocked\':[[onset,cutoff]],\'start_index\':start,\'onset_s\':onset,\n                      \'cutoff_s\':cutoff,\'duration_s\':duration})\n    order=rng.permutation(len(specs));return [specs[i] for i in order]\n\n\ndef eval_prediction(model,inputs,device,zero_imu=False):\n    imu=inputs[\'imu_features\']\n    if zero_imu:imu=np.zeros_like(imu)\n    model.eval()\n    with torch.inference_mode():\n        dv,_=model(tensor(imu,device)[None],tensor(inputs[\'gnss_features\'],device)[None])\n        motion=rollout_motion(dv,tensor(inputs[\'velocity_prior\'],device)[None],\n            tensor(inputs[\'integration_dt\'],device)[None],tensor(inputs[\'availability\'],device,torch.bool)[None])\n    return inputs[\'held\']+motion[0].cpu().numpy(),motion[0].cpu().numpy()\n\n\ndef execute(config_path,raw_dir,out_dir,run_training=False):\n    if not run_training:\n        raise RuntimeError(\'Pilot is prepared only. Separate user command is required; pass --run-training afterward.\')\n    config_path=Path(config_path);cfg=json.loads(config_path.read_text())\n    plan=json.loads(config_path.with_name(\'v2_motion_pilot_manifest.json\').read_text())\n    if hashlib.sha256(config_path.read_bytes()).hexdigest()!=plan[\'config_sha256\']:\n        raise ValueError(\'pilot config differs from prepared manifest\')\n    out=Path(out_dir)\n    if out.exists():raise FileExistsError(\'no overwrite/resume: \'+str(out))\n    data=prepare(raw_dir,cfg)\n    scaler=fit_imu_scaler(data[\'imu\'],cfg[\'train_stop_index\'])\n    ids=np.arange(*cfg[\'validation_common_indices\'])\n    if len(ids)!=2688:raise ValueError(\'unexpected common validation support\')\n    # Validate every planned scenario and label support BEFORE optimizing.\n    prepared=[]\n    for s in plan[\'scenarios\']:\n        mask=availability_from_blocks(len(ids),s[\'unavailable_index_runs\'])\n        if s[\'requested_duration_s\']:\n            actual=(data[\'t\'][ids]>=s[\'onset_s\'])&(data[\'t\'][ids]<s[\'requested_cutoff_s\'])\n            if not np.array_equal(actual,~mask):raise ValueError(\'fixed validation timestamps changed\')\n        x=build_inputs(data,ids,mask,scaler,s[\'source_blocked_intervals_s\'])\n        labels,label_mask,gt=motion_labels(data,x)\n        prepared.append((s,x,labels,gt))\n    for e in range(cfg[\'epochs\']):\n        for spec in training_specs(data,cfg,e):\n            x=build_inputs(data,spec[\'indices\'],spec[\'available\'],scaler,spec[\'blocked\'])\n            motion_labels(data,x)\n    out.mkdir(parents=True)\n    (out/\'checkpoints\').mkdir()\n    # Preserve the exact implementation actually used by the future pilot.\n    code_root=Path(__file__).resolve().parents[2]\n    source_hashes={}\n    for relative in [\'src/pilot_v2/data.py\',\'src/pilot_v2/model.py\',\'src/pilot_v2/run.py\',\n                     \'src/diagnostics/clean_outages.py\']:\n        payload=(code_root/relative).read_bytes()\n        destination=out/\'source_snapshot\'/relative\n        destination.parent.mkdir(parents=True,exist_ok=True)\n        destination.write_bytes(payload)\n        source_hashes[relative]=hashlib.sha256(payload).hexdigest()\n    json_write(out/\'source_sha256.json\',source_hashes)\n    json_write(out/\'status.json\',{\'status\':\'running\',\'seed\':cfg[\'seed\'],\'test_used\':False})\n    json_write(out/\'config.json\',cfg);json_write(out/\'manifest.json\',plan);json_write(out/\'imu_scaler_train_only.json\',scaler)\n    json_write(out/\'coordinate_contract.json\',{\'origin\':data[\'origin_utm_e_n_msl_m\'],\'note\':data[\'frame_note\'],\n                                             \'target\':\'camera relative motion; no fit/alignment of GT to inputs\'})\n    device=\'cuda\' if torch.cuda.is_available() else \'cpu\'\n    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False\n    torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True\n    import pandas,pyproj\n    json_write(out/\'environment.json\',{\'python\':platform.python_version(),\'torch\':torch.__version__,\n        \'numpy\':np.__version__,\'pandas\':pandas.__version__,\'pyproj\':pyproj.__version__,\n        \'device\':device,\'cuda\':torch.version.cuda,\n        \'device_name\':torch.cuda.get_device_name(0) if device==\'cuda\' else platform.processor()})\n    models={};history=[];augmentation=[];count={}\n    try:\n        for variant in cfg[\'methods\']:\n            torch.manual_seed(cfg[\'seed\']);np.random.seed(cfg[\'seed\'])\n            model=CausalMotion(cfg[\'hidden\'],variant[\'ablation\'],variant[\'backbone\']).to(device)\n            count[variant[\'id\']]=sum(p.numel() for p in model.parameters())\n            optimizer=torch.optim.AdamW(model.parameters(),lr=cfg[\'learning_rate\'],weight_decay=cfg[\'weight_decay\'])\n            for epoch in range(cfg[\'epochs\']):\n                model.train();specs=training_specs(data,cfg,epoch);losses=[]\n                for offset in range(0,len(specs),cfg[\'batch_size\']):\n                    selected=specs[offset:offset+cfg[\'batch_size\']]\n                    xx=[];yy=[];mm=[]\n                    for spec in selected:\n                        x=build_inputs(data,spec[\'indices\'],spec[\'available\'],scaler,spec[\'blocked\'])\n                        y,m,_=motion_labels(data,x)\n                        xx.append(x);yy.append(y);mm.append(m)\n                        if variant==cfg[\'methods\'][0]:\n                            augmentation.append({\'epoch\':epoch,**{k:v for k,v in spec.items() if k not in (\'indices\',\'available\',\'blocked\')}})\n                    stack=lambda key:tensor(np.stack([x[key] for x in xx]),device)\n                    availability=stack(\'availability\').bool()\n                    dv,_=model(stack(\'imu_features\'),stack(\'gnss_features\'))\n                    motion=rollout_motion(dv,stack(\'velocity_prior\'),stack(\'integration_dt\'),availability)\n                    chosen=tensor(np.stack(mm),device,torch.bool)\n                    target=tensor(np.stack(yy),device)\n                    loss=torch.nn.functional.huber_loss(motion[chosen],target[chosen],delta=cfg[\'huber_delta_m\'])\n                    if not torch.isfinite(loss):raise ValueError(\'nonfinite loss\')\n                    optimizer.zero_grad();loss.backward()\n                    torch.nn.utils.clip_grad_norm_(model.parameters(),cfg[\'grad_clip\']);optimizer.step()\n                    losses.append(float(loss.detach().cpu()))\n                history.append({\'method\':variant[\'id\'],\'epoch\':epoch+1,\'mean_batch_training_loss\':float(np.mean(losses))})\n                print(history[-1],flush=True)\n            checkpoint=out/\'checkpoints\'/(variant[\'id\']+\'_seed0.pt\')\n            torch.save(model.state_dict(),checkpoint)\n            models[variant[\'id\']]=model.eval()\n        json_write(out/\'parameter_counts.json\',count)\n        json_write(out/\'training_augmentation_manifest.json\',augmentation)\n        csv_write(out/\'training_history.csv\',history)\n        rows=[]\n        for scenario,x,target,gt in prepared:\n            sid=scenario[\'scenario_id\'];folder=out/sid;folder.mkdir()\n            held,cv=baseline_predictions(x)\n            predictions={\'held_gnss\':held,\'constant_velocity_gnss\':cv}\n            for mid,model in models.items():predictions[mid]=eval_prediction(model,x,device)[0]\n            predictions[\'fusion_v2_full_zero_imu\']=eval_prediction(models[\'fusion_v2_full\'],x,device,True)[0]\n            # Primary motion metric removes initial fix error ONLY in labels, not model state.\n            mask=x[\'availability\'];anchor_base=x[\'held\'].copy()\n            for a,b in false_runs(mask):anchor_base[a:b]=held[a-1]\n            for mid,pred in predictions.items():\n                metric=raw_metrics(pred,gt,mask)\n                motion_error=(pred-anchor_base)-target\n                selected=~mask\n                motion_rmse=float(np.sqrt(np.mean(np.sum(motion_error[selected]**2,axis=1)))) if selected.any() else None\n                checkpoint=(out/\'checkpoints\'/((mid if mid!=\'fusion_v2_full_zero_imu\' else \'fusion_v2_full\')+\'_seed0.pt\'))\n                checkpoint_hash=hashlib.sha256(checkpoint.read_bytes()).hexdigest() if checkpoint.exists() else None\n                meta={\'scenario\':scenario,\'method\':mid,\'seed\':cfg[\'seed\'] if checkpoint_hash else None,\n                      \'checkpoint_sha256\':checkpoint_hash,\'frame\':data[\'frame_note\'],\n                      \'target\':\'relative camera motion from last legal fix timestamp; no GT input/state\',\n                      \'non_independent_validation\':True,\'zero_imu_is_sensitivity\':mid.endswith(\'zero_imu\')}\n                np.savez_compressed(folder/(mid+\'.npz\'),pred=pred,gt=gt,timestamps=x[\'t\'],\n                    availability=mask,motion_target=target,motion_error=motion_error,\n                    source_timestamp=x[\'clean\'][\'held_source_timestamp_s\'],\n                    source_index=x[\'clean\'][\'held_source_index\'],metadata_json=np.array(json.dumps(meta)))\n                rows.append({\'scenario_id\':sid,\'method\':mid,\'seed\':meta[\'seed\'],\n                             \'checkpoint_sha256\':checkpoint_hash,\'motion_rmse_3d_m\':motion_rmse,**metric})\n        csv_write(out/\'summary.csv\',rows)\n        json_write(out/\'status.json\',{\'status\':\'complete\',\'seed\':cfg[\'seed\'],\'evaluation_rows\':len(rows),\n                                      \'checkpoint_selection\':\'fixed final epoch, no validation selection\',\'test_used\':False})\n    except Exception as error:\n        json_write(out/\'status.json\',{\'status\':\'failed\',\'error\':str(error)});raise\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--config\',required=True);parser.add_argument(\'--raw-dir\',required=True)\n    parser.add_argument(\'--out-dir\',default=\'outputs/v2_motion_pilot\')\n    parser.add_argument(\'--run-training\',action=\'store_true\')\n    a=parser.parse_args();execute(a.config,a.raw_dir,a.out_dir,a.run_training)\n', 'tests/test_next_stage_v2.py': '"""Small deterministic numerical/causality tests; never train any model."""\nimport sys\nimport unittest\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]/\'src\'))\nfrom diagnostics.next_stage import fit_frame,apply_frame,utm32_wgs84,future_gnss_exposure,permute_imu_windows\nfrom diagnostics.clean_outages import availability_from_blocks\nfrom pilot_v2.data import backward_indices,build_inputs,motion_labels,fit_imu_scaler,baseline_predictions\nfrom pilot_v2.model import CausalMotion,rollout_motion\nfrom pilot_v2.run import execute,training_specs\n\n\nclass GeometryTests(unittest.TestCase):\n    def test_rigid_recovery_and_no_scale(self):\n        x=np.random.default_rng(1).normal(size=(30,3))\n        angle=.3;r=np.array([[np.cos(angle),-np.sin(angle),0],[np.sin(angle),np.cos(angle),0],[0,0,1]])\n        y=np.einsum(\'ni,ji->nj\',x,r)+[3,-2,7]\n        for kind in [\'se3\',\'se2_vertical\']:\n            fitted=fit_frame(x,y,kind)\n            np.testing.assert_allclose(apply_frame(x,fitted),y,atol=1e-12)\n            self.assertEqual(fitted[\'scale\'],1.)\n\n    def test_similarity_is_explicit_and_translation_only(self):\n        x=np.random.default_rng(2).normal(size=(30,3));y=2*x+[1,2,3]\n        f=fit_frame(x,y,\'similarity_diagnostic\')\n        self.assertAlmostEqual(f[\'scale\'],2)\n        np.testing.assert_allclose(apply_frame(x,f),y,atol=1e-12)\n        f=fit_frame(x,x+[1,2,3],\'translation\')\n        np.testing.assert_allclose(f[\'translation_m\'],[1,2,3],atol=1e-12)\n\n    def test_projection_reference_points(self):\n        np.testing.assert_allclose(utm32_wgs84([0],[9]),[[500000,0]],atol=1e-6)\n        # Independent published dataset GPS UTM first point (rounding < 1 mm).\n        np.testing.assert_allclose(utm32_wgs84([47.3843571],[8.5451784]),\n                                   [[465670.706847,5247978.033438]],atol=.001,rtol=0)\n\n    def test_future_exposure_exact_windows(self):\n        t=np.arange(12.);mask=availability_from_blocks(12,[(3,9)])\n        exposed,fresh=future_gnss_exposure(t,mask,t,9.,4)\n        np.testing.assert_array_equal(np.flatnonzero(exposed),[8])\n        np.testing.assert_array_equal(exposed,fresh)\n\n    def test_mask_recovery_on_window_boundary_has_no_exposure(self):\n        t=np.arange(12.);mask=availability_from_blocks(12,[(2,8)])\n        exposed,_=future_gnss_exposure(t,mask,t,8.,4)\n        self.assertFalse(exposed.any())\n\n    def test_imu_permutation_preserves_joint_rows_and_windows(self):\n        x=np.arange(48).reshape(8,6)\n        for mode in [\'shuffle\',\'reverse\']:\n            out,order=permute_imu_windows(x,4,mode,17)\n            for w in range(2):\n                np.testing.assert_array_equal(out[4*w:4*w+4],x[4*w+order[w]])\n            np.testing.assert_array_equal(out,permute_imu_windows(x,4,mode,17)[0])\n\n\nclass PilotDataTests(unittest.TestCase):\n    def setUp(self):\n        self.t=np.arange(80,dtype=float)\n        self.et=np.arange(-.2,81,2.)\n        self.data={\'t\':self.t,\'imu\':np.tile(np.arange(6.),(80,1)), \'gyro_age\':np.full(80,.01),\n                   \'event_t\':self.et,\'event_position\':np.column_stack([2*self.et,-self.et,.5*self.et]),\n                   \'gt_t\':np.array([-1.,81.]),\'gt_position\':np.array([[-2.,1.,-.5],[162.,-81.,40.5]])}\n        self.mask=availability_from_blocks(80,[(30,60)])\n        self.scaler={\'mean\':[0.]*6,\'std\':[1.]*6}\n\n    def inputs(self,data=None):\n        return build_inputs(self.data if data is None else data,np.arange(80),self.mask,self.scaler,[[30.,60.]])\n\n    def test_backward_sync_does_not_choose_future_and_rejects_no_history(self):\n        np.testing.assert_array_equal(backward_indices(np.array([0.,.09,.19]),np.array([.1,.2])),[1,2])\n        with self.assertRaises(ValueError):backward_indices(np.array([.1]),np.array([0.]))\n\n    def test_scaler_train_only(self):\n        a=np.arange(120.).reshape(20,6);b=a.copy();b[10:]=1e9\n        self.assertEqual(fit_imu_scaler(a,10),fit_imu_scaler(b,10))\n\n    def test_inputs_independent_of_gt_and_absolute_translation(self):\n        original=self.inputs();changed=dict(self.data)\n        changed[\'gt_position\']=self.data[\'gt_position\']+1e9\n        changed[\'event_position\']=self.data[\'event_position\']+[1000,-2000,500]\n        alternative=self.inputs(changed)\n        np.testing.assert_allclose(original[\'gnss_features\'],alternative[\'gnss_features\'],atol=1e-5)\n        np.testing.assert_array_equal(original[\'imu_features\'],alternative[\'imu_features\'])\n\n    def test_prefix_inputs_unchanged_by_future_sensor_values(self):\n        original=self.inputs();changed=dict(self.data)\n        changed[\'event_position\']=self.data[\'event_position\'].copy()\n        changed[\'event_position\'][self.et>65]+=1e6\n        changed[\'imu\']=self.data[\'imu\'].copy();changed[\'imu\'][65:]+=100\n        alternative=self.inputs(changed)\n        np.testing.assert_array_equal(original[\'gnss_features\'][:65],alternative[\'gnss_features\'][:65])\n        np.testing.assert_array_equal(original[\'imu_features\'][:65],alternative[\'imu_features\'][:65])\n\n    def test_no_gnss_updates_inside_outage_and_anchor_dt(self):\n        x=self.inputs()\n        self.assertFalse(x[\'clean\'][\'gnss_update_mask\'][30:60].any())\n        self.assertAlmostEqual(x[\'integration_dt\'][30],self.t[30]-x[\'clean\'][\'held_source_timestamp_s\'][29],places=5)\n\n    def test_motion_labels_invariant_to_constant_gt_origin(self):\n        x=self.inputs();target,mask,_=motion_labels(self.data,x)\n        changed=dict(self.data);changed[\'gt_position\']=self.data[\'gt_position\']+[10,20,-30]\n        y,m,_=motion_labels(changed,x)\n        np.testing.assert_allclose(target,y,atol=1e-6);np.testing.assert_array_equal(mask,m)\n        self.assertFalse(mask[self.mask].any())\n\n    def test_zero_velocity_head_reproduces_cv_motion(self):\n        x=self.inputs();held,cv=baseline_predictions(x)\n        motion=rollout_motion(torch.zeros(1,80,3),torch.tensor(x[\'velocity_prior\'])[None],\n            torch.tensor(x[\'integration_dt\'])[None],torch.tensor(self.mask)[None]).numpy()[0]\n        np.testing.assert_allclose(held+motion,cv,atol=3e-5)\n\n    def test_training_augmentation_is_deterministic(self):\n        cfg={\'seed\':0,\'train_window_samples\':1024,\'train_stride_samples\':512,\'train_stop_index\':3000,\n             \'outage_durations_s\':[10,30,60],\'train_warmup_s\':30,\'train_recovery_s\':5}\n        a=training_specs({\'t\':np.arange(3000)*.1},cfg,2)\n        b=training_specs({\'t\':np.arange(3000)*.1},cfg,2)\n        for x,y in zip(a,b):\n            self.assertEqual(x[\'onset_s\'],y[\'onset_s\']);np.testing.assert_array_equal(x[\'available\'],y[\'available\'])\n\n\nclass ModelTests(unittest.TestCase):\n    def test_causal_prefix_and_state_carry_with_nonzero_head(self):\n        torch.set_num_threads(1);torch.manual_seed(5)\n        for backbone in [\'gru\',\'lstm\']:\n            model=CausalMotion(backbone=backbone).eval();torch.nn.init.normal_(model.head.weight)\n            i=torch.randn(1,20,8);g=torch.randn(1,20,10)\n            with torch.no_grad():\n                full,_=model(i,g)\n                mutated=i.clone();mutated[:,10:]+=10\n                altered,_=model(mutated,g)\n                torch.testing.assert_close(full[:,:10],altered[:,:10])\n                self.assertGreater(float((full[:,10:]-altered[:,10:]).abs().max()),1e-4)\n                first,state=model(i[:,:10],g[:,:10]);second,_=model(i[:,10:],g[:,10:],state)\n                torch.testing.assert_close(full,torch.cat([first,second],1))\n\n    def test_matched_parameter_count_and_gps_only_ignores_all_imu_features(self):\n        full=CausalMotion();gps=CausalMotion(ablation=\'gps_only\')\n        self.assertEqual(sum(p.numel() for p in full.parameters()),sum(p.numel() for p in gps.parameters()))\n        torch.nn.init.normal_(gps.head.weight)\n        i=torch.randn(1,10,8);g=torch.randn(1,10,10)\n        with torch.no_grad():torch.testing.assert_close(gps(i,g)[0],gps(i*100,g)[0])\n\n    def test_training_guard_before_any_io(self):\n        with self.assertRaisesRegex(RuntimeError,\'Separate user command\'):\n            execute(\'/does/not/exist\',\'/not/raw\',\'/not/output\',run_training=False)\n\n\nif __name__==\'__main__\':unittest.main()\n', 'configs/v2_motion_pilot.json': '{\n  "pilot_id": "fusionnav_v2_causal_motion_pilot_v1",\n  "prepared_only": true,\n  "seed": 0,\n  "raw_dataset": "mrisdal/zurich-urban-micro-aerial-vehicle",\n  "raw_dataset_version": 2,\n  "raw_sha256": {\n    "RawAccel.csv": "2609d0d936c75c7ed2566b258f87c7508faa28848e5cd366f55fbf069e557c11",\n    "RawGyro.csv": "3d39a569265dcc9682fe254c5246a7b7b84c14f338c29a99f25d435d683d7bec",\n    "OnboardGPS.csv": "8e58193ea9ff4e085a61241a66179d46f66110074d59cc1b97a7167fca1181a5",\n    "GroundTruthAGL.csv": "50e95abc70461567a915a76ba17231ae3088b834dbcb433737135eacb9c9c552"\n  },\n  "coordinate_contract": "WGS84 UTM32N for both position streams; same first valid GNSS XYZ origin; z retained as documented metres above MSL for GPS; no fitted alignment",\n  "unresolved_geometry": [\n    "precise GT vertical datum/geoid metadata",\n    "GNSS antenna extrinsic; camera-body offset alone insufficient"\n  ],\n  "primary_evaluation": "relative camera motion from last legal GNSS source timestamp; constant position offsets cancel in label only",\n  "secondary_evaluation": "raw absolute camera-position error, with existing GNSS anchor error; no alignment",\n  "features": "8 IMU features; 10 causal GNSS/time features; no absolute XYZ, absolute flight time, onboard pose, GT input or GT reset",\n  "gnss_epoch_policy": "stable timestamp sort; deduplicate timestamps; coordinate-change events with fix_type>=3 are proxies for physical fixes",\n  "train_stop_index": 18935,\n  "validation_original_indices": [\n    18935,\n    21640\n  ],\n  "validation_common_indices": [\n    18935,\n    21623\n  ],\n  "evaluation_state": "continuous causal recurrent state across entire validation scenario; reset only at scenario start; availability recovery resets position to legal fix",\n  "epochs": 8,\n  "checkpoint_selection": "fixed final epoch; no best-run or best-validation selection",\n  "hidden": 32,\n  "learning_rate": 0.001,\n  "weight_decay": 0.0001,\n  "batch_size": 4,\n  "grad_clip": 1.0,\n  "huber_delta_m": 1.0,\n  "loss": "Huber on integrated relative displacement from native GT interpolation, selected about every real second during outage; no absolute GNSS-to-GT position target",\n  "train_window_samples": 1024,\n  "train_stride_samples": 512,\n  "train_warmup_s": 30,\n  "train_recovery_s": 5,\n  "outage_durations_s": [\n    10,\n    30,\n    60\n  ],\n  "augmentation_seed_rule": "seed + 10000 + epoch; identical windows/masks/order across full and gps_only",\n  "methods": [\n    {\n      "id": "fusion_v2_full",\n      "ablation": "full",\n      "backbone": "gru"\n    },\n    {\n      "id": "fusion_v2_gps_only",\n      "ablation": "gps_only",\n      "backbone": "gru"\n    },\n    {\n      "id": "lstm_v2_full",\n      "ablation": "full",\n      "backbone": "lstm"\n    }\n  ],\n  "go_to_five_seeds": "consistent practically relevant paired improvement versus gps_only on multiple fixed episodes, especially 30/60 s; inspect sensitivity and CV comparison; no mandatory percentage",\n  "execution_authorization": "requires separate user command; notebook RUN_TRAINING defaults False",\n  "train_onset_margin": "reserve two maximum observed sample gaps for onset/recovery snapping"\n}\n', 'configs/v2_motion_pilot_manifest.json': '{\n  "pilot_id": "fusionnav_v2_causal_motion_pilot_v1",\n  "status": "prepared_not_trained",\n  "config_sha256": "f413f38984fd816e6ade634dcf329488c3225117085f7116f041c1622e43c6b0",\n  "seed": 0,\n  "scenarios": [\n    {\n      "scenario_id": "val_control",\n      "split": "validation",\n      "episode_id": null,\n      "requested_duration_s": 0,\n      "source_blocked_intervals_s": [],\n      "unavailable_index_runs": [],\n      "n_unavailable": 0,\n      "metric_scope": "whole_common_trace",\n      "n_common": 2688,\n      "mask_uint8_sha256": "588b1bea0cec08685bbc845b232ccb924e129421942bd1ab3065b6e7f79a3bef",\n      "generator_seed": 0,\n      "rng_used": false\n    },\n    {\n      "scenario_id": "val_e1_d10s",\n      "split": "validation",\n      "episode_id": 1,\n      "requested_duration_s": 10,\n      "target_onset_s": 1975.82550675,\n      "onset_s": 1975.869753,\n      "requested_cutoff_s": 1985.869753,\n      "last_unavailable_s": 1985.7717619999999,\n      "first_link_available_after_s": 1985.871752,\n      "observed_mask_interval_s": 10.001999000000069,\n      "unavailable_sample_span_s": 9.902008999999907,\n      "available_history_span_s": 70.11699599999997,\n      "recovery_span_s": 190.17200400000002,\n      "source_blocked_intervals_s": [\n        [\n          1975.869753,\n          1985.869753\n        ]\n      ],\n      "unavailable_index_runs": [\n        [\n          686,\n          786\n        ]\n      ],\n      "n_unavailable": 100,\n      "metric_scope": "outage_only",\n      "n_common": 2688,\n      "mask_uint8_sha256": "9478793466ccff3f5a44e05df5b253ad2f7cc6695f5487567ebcd1da995cbd7d",\n      "generator_seed": 0,\n      "rng_used": false\n    },\n    {\n      "scenario_id": "val_e1_d30s",\n      "split": "validation",\n      "episode_id": 1,\n      "requested_duration_s": 30,\n      "target_onset_s": 1975.82550675,\n      "onset_s": 1975.869753,\n      "requested_cutoff_s": 2005.869753,\n      "last_unavailable_s": 2005.7787529999998,\n      "first_link_available_after_s": 2005.878753,\n      "observed_mask_interval_s": 30.009000000000015,\n      "unavailable_sample_span_s": 29.908999999999878,\n      "available_history_span_s": 70.11699599999997,\n      "recovery_span_s": 170.16500300000007,\n      "source_blocked_intervals_s": [\n        [\n          1975.869753,\n          2005.869753\n        ]\n      ],\n      "unavailable_index_runs": [\n        [\n          686,\n          986\n        ]\n      ],\n      "n_unavailable": 300,\n      "metric_scope": "outage_only",\n      "n_common": 2688,\n      "mask_uint8_sha256": "7d0af73a47f34df6d72982ada3badbd63efe284f363d489628995b4f3afd9881",\n      "generator_seed": 0,\n      "rng_used": false\n    },\n    {\n      "scenario_id": "val_e1_d60s",\n      "split": "validation",\n      "episode_id": 1,\n      "requested_duration_s": 60,\n      "target_onset_s": 1975.82550675,\n      "onset_s": 1975.869753,\n      "requested_cutoff_s": 2035.869753,\n      "last_unavailable_s": 2035.788752,\n      "first_link_available_after_s": 2035.888753,\n      "observed_mask_interval_s": 60.019000000000005,\n      "unavailable_sample_span_s": 59.918998999999985,\n      "available_history_span_s": 70.11699599999997,\n      "recovery_span_s": 140.15500300000008,\n      "source_blocked_intervals_s": [\n        [\n          1975.869753,\n          2035.869753\n        ]\n      ],\n      "unavailable_index_runs": [\n        [\n          686,\n          1286\n        ]\n      ],\n      "n_unavailable": 600,\n      "metric_scope": "outage_only",\n      "n_common": 2688,\n      "mask_uint8_sha256": "0b086cc043ee5ea7afba90e7fadd076edd9bd6b2a0d2f616dbb5d05f011e7724",\n      "generator_seed": 0,\n      "rng_used": false\n    },\n    {\n      "scenario_id": "val_e2_d10s",\n      "split": "validation",\n      "episode_id": 2,\n      "requested_duration_s": 10,\n      "target_onset_s": 2015.8982565000001,\n      "onset_s": 2015.9817529999998,\n      "requested_cutoff_s": 2025.9817529999998,\n      "last_unavailable_s": 2025.8857529999998,\n      "first_link_available_after_s": 2025.986754,\n      "observed_mask_interval_s": 10.00500100000022,\n      "unavailable_sample_span_s": 9.903999999999996,\n      "available_history_span_s": 110.22899599999982,\n      "recovery_span_s": 150.057002,\n      "source_blocked_intervals_s": [\n        [\n          2015.9817529999998,\n          2025.9817529999998\n        ]\n      ],\n      "unavailable_index_runs": [\n        [\n          1087,\n          1187\n        ]\n      ],\n      "n_unavailable": 100,\n      "metric_scope": "outage_only",\n      "n_common": 2688,\n      "mask_uint8_sha256": "f872ab438fa975ee018f3d6c65dd8caa39fc890e192c83c75c42d8209d0f0f0b",\n      "generator_seed": 0,\n      "rng_used": false\n    },\n    {\n      "scenario_id": "val_e2_d30s",\n      "split": "validation",\n      "episode_id": 2,\n      "requested_duration_s": 30,\n      "target_onset_s": 2015.8982565000001,\n      "onset_s": 2015.9817529999998,\n      "requested_cutoff_s": 2045.9817529999998,\n      "last_unavailable_s": 2045.893754,\n      "first_link_available_after_s": 2045.993796,\n      "observed_mask_interval_s": 30.012043000000176,\n      "unavailable_sample_span_s": 29.912001000000146,\n      "available_history_span_s": 110.22899599999982,\n      "recovery_span_s": 130.04996000000006,\n      "source_blocked_intervals_s": [\n        [\n          2015.9817529999998,\n          2045.9817529999998\n        ]\n      ],\n      "unavailable_index_runs": [\n        [\n          1087,\n          1387\n        ]\n      ],\n      "n_unavailable": 300,\n      "metric_scope": "outage_only",\n      "n_common": 2688,\n      "mask_uint8_sha256": "29d5854659ff5815929f2757a901e6ec74ef2f31a39ca9f9ced5a7f01b969755",\n      "generator_seed": 0,\n      "rng_used": false\n    },\n    {\n      "scenario_id": "val_e2_d60s",\n      "split": "validation",\n      "episode_id": 2,\n      "requested_duration_s": 60,\n      "target_onset_s": 2015.8982565000001,\n      "onset_s": 2015.9817529999998,\n      "requested_cutoff_s": 2075.981753,\n      "last_unavailable_s": 2075.9067919999998,\n      "first_link_available_after_s": 2076.006753,\n      "observed_mask_interval_s": 60.02500000000032,\n      "unavailable_sample_span_s": 59.92503899999997,\n      "available_history_span_s": 110.22899599999982,\n      "recovery_span_s": 100.03700299999991,\n      "source_blocked_intervals_s": [\n        [\n          2015.9817529999998,\n          2075.981753\n        ]\n      ],\n      "unavailable_index_runs": [\n        [\n          1087,\n          1687\n        ]\n      ],\n      "n_unavailable": 600,\n      "metric_scope": "outage_only",\n      "n_common": 2688,\n      "mask_uint8_sha256": "4f34e8d92654f4db91151dabf1fca1c766b9ecc507dab76928cbaee378224c20",\n      "generator_seed": 0,\n      "rng_used": false\n    },\n    {\n      "scenario_id": "val_e3_d10s",\n      "split": "validation",\n      "episode_id": 3,\n      "requested_duration_s": 10,\n      "target_onset_s": 2055.97100625,\n      "onset_s": 2055.997753,\n      "requested_cutoff_s": 2065.997753,\n      "last_unavailable_s": 2065.903754,\n      "first_link_available_after_s": 2066.003758,\n      "observed_mask_interval_s": 10.006004999999732,\n      "unavailable_sample_span_s": 9.906000999999833,\n      "available_history_span_s": 150.24499600000013,\n      "recovery_span_s": 110.0399980000002,\n      "source_blocked_intervals_s": [\n        [\n          2055.997753,\n          2065.997753\n        ]\n      ],\n      "unavailable_index_runs": [\n        [\n          1487,\n          1587\n        ]\n      ],\n      "n_unavailable": 100,\n      "metric_scope": "outage_only",\n      "n_common": 2688,\n      "mask_uint8_sha256": "0074eb8d8dc7bc8e4228c9480866b762e70819a1287206fa3700ff486e693957",\n      "generator_seed": 0,\n      "rng_used": false\n    },\n    {\n      "scenario_id": "val_e3_d30s",\n      "split": "validation",\n      "episode_id": 3,\n      "requested_duration_s": 30,\n      "target_onset_s": 2055.97100625,\n      "onset_s": 2055.997753,\n      "requested_cutoff_s": 2085.997753,\n      "last_unavailable_s": 2085.910765,\n      "first_link_available_after_s": 2086.010757,\n      "observed_mask_interval_s": 30.01300399999991,\n      "unavailable_sample_span_s": 29.91301199999998,\n      "available_history_span_s": 150.24499600000013,\n      "recovery_span_s": 90.03299900000002,\n      "source_blocked_intervals_s": [\n        [\n          2055.997753,\n          2085.997753\n        ]\n      ],\n      "unavailable_index_runs": [\n        [\n          1487,\n          1787\n        ]\n      ],\n      "n_unavailable": 300,\n      "metric_scope": "outage_only",\n      "n_common": 2688,\n      "mask_uint8_sha256": "c42dd6904e081422b95fc0d90f246f1dfddf40104196dc8094da0e0c6cb7178d",\n      "generator_seed": 0,\n      "rng_used": false\n    },\n    {\n      "scenario_id": "val_e3_d60s",\n      "split": "validation",\n      "episode_id": 3,\n      "requested_duration_s": 60,\n      "target_onset_s": 2055.97100625,\n      "onset_s": 2055.997753,\n      "requested_cutoff_s": 2115.997753,\n      "last_unavailable_s": 2115.9237519999997,\n      "first_link_available_after_s": 2116.023753,\n      "observed_mask_interval_s": 60.02599999999984,\n      "unavailable_sample_span_s": 59.92599899999959,\n      "available_history_span_s": 150.24499600000013,\n      "recovery_span_s": 60.02000300000009,\n      "source_blocked_intervals_s": [\n        [\n          2055.997753,\n          2115.997753\n        ]\n      ],\n      "unavailable_index_runs": [\n        [\n          1487,\n          2087\n        ]\n      ],\n      "n_unavailable": 600,\n      "metric_scope": "outage_only",\n      "n_common": 2688,\n      "mask_uint8_sha256": "a6662edb1addc52e5cd4647169edf53c5742e2106643354ecb051cd3a83ee997",\n      "generator_seed": 0,\n      "rng_used": false\n    }\n  ],\n  "common_timestamps_float64_le_sha256": "65f65ce416cd8b5a2b4d4e89fefb8b81cea2da9ecf5149d38e4c637eb5d1b110",\n  "mask_comparison": "same fixed timestamps/availability as diagnostic v1; GNSS preprocessing and causal model/target change; not directly comparable to R7 as an isolated architecture effect",\n  "planned_training_runs": 3,\n  "planned_neural_seeds": 1,\n  "test_split_used": false,\n  "training_executed": false\n}\n'}
HASHES = {'src/diagnostics/__init__.py': 'fd3a6eb7a18cd1cc76c1ac952d11562fc8a6ca0744f9dae7fc405954e09f3894', 'src/diagnostics/clean_outages.py': 'f86dfb712be86f6c1a2c03d18d77cb4800339337fdd57924d7cee8d5ec9b80b4', 'src/diagnostics/next_stage.py': '9ea21e717f5b9d751464334475611e3951b33990ecd0d37919b332ad42737a88', 'src/pilot_v2/__init__.py': 'a8e1cc4fd921c07d987e67e0873cff4aa437eca63bdcc7b25d0d6f5719f8fb19', 'src/pilot_v2/data.py': 'b1579f8a4eb148530a8ce4b63ddd6ab69424ce38851088744c56451940f05102', 'src/pilot_v2/model.py': 'ba51930e2a9c3383cad82f874d82c31306726e58a81112f08ab3aa8ba9633bd6', 'src/pilot_v2/run.py': '8f8d389592334c29726108782708285b99f94e6e5def45e39b942beffddf4877', 'tests/test_next_stage_v2.py': '357902d9e97556c78950980cfb5d8bedb5d5d8aa15b2b615b98c501673e82bd6', 'configs/v2_motion_pilot.json': 'f413f38984fd816e6ade634dcf329488c3225117085f7116f041c1622e43c6b0', 'configs/v2_motion_pilot_manifest.json': '37b1837afb89f371d291125d4f35b26782a8353375e1dc213211e3f670e70900'}
for name,source in SOURCES.items():
    payload=source.encode();assert hashlib.sha256(payload).hexdigest()==HASHES[name]
    path=CODE_ROOT/name
    if path.exists() and path.read_bytes()!=payload:raise RuntimeError('Use a fresh session: '+str(path))
    path.parent.mkdir(parents=True,exist_ok=True)
    if not path.exists():path.write_bytes(payload)
sys.path.insert(0,str(CODE_ROOT/'src'))


In [ ]:
import importlib.util
import subprocess
required=['numpy','pandas','torch','pyproj']
missing=[p for p in required if importlib.util.find_spec(p) is None]
if missing:raise RuntimeError(f'Required packages absent: {missing}; no installation was attempted')
subprocess.run([sys.executable,'-B','-m','unittest','discover','-s',str(CODE_ROOT/'tests'),
                '-p','test_next_stage_v2.py','-v'],cwd=CODE_ROOT,check=True)


In [ ]:
import json
config_path=CODE_ROOT/'configs/v2_motion_pilot.json'
config=json.loads(config_path.read_text())
if RAW_DIR is None:
    candidates=[]
    for p in Path('/kaggle/input').rglob('RawAccel.csv'):
        root=p.parent
        if all((root/n).is_file() for n in config['raw_sha256']):
            if all(hashlib.sha256((root/n).read_bytes()).hexdigest()==h for n,h in config['raw_sha256'].items()):
                candidates.append(root)
    if len(candidates)!=1:raise RuntimeError(f'Expected exactly one matching raw dataset mount, found {len(candidates)}; set RAW_DIR explicitly')
    RAW_DIR=candidates[0]
print('Pinned raw input:',RAW_DIR)
print('Prepared only:',not RUN_TRAINING)
print('Plan: 3 models, seed 0, 8 epochs, 10 fixed validation scenarios, no test')


In [ ]:
from pilot_v2.run import execute
if RUN_TRAINING:
    execute(config_path,RAW_DIR,OUT_DIR,run_training=True)
else:
    print('TRAINING NOT STARTED. Wait for the separate user command before enabling RUN_TRAINING.')


## После разрешённого запуска

Верните весь `outputs/v2_motion_pilot/`: config/manifest, normalization, training history,
checkpoints, summary.csv, status.json и предсказания всех методов/сценариев.
Сравните full и gps_only по paired motion RMSE на 30/60 с, absolute raw error отдельно,
и sensitivity того же full checkpoint к нулевому IMU. Сопоставьте с held/CV.
Улучшение должно повторяться на нескольких фиксированных эпизодах и быть практически
существенным; обязательный процент не задан. Переход к 5 seeds не автоматический.
R7 и прежние diagnostic outputs не перезаписываются.
